# Notebook 01 — Data Loading & Merging

**Purpose:** Load all 20 USER_*.csv files from `data/raw/`, extract the correct columns, merge into a single clean DataFrame, and save to `data/processed/cleaned_dataset.csv`.

**Column layout (no headers in source files):**
- Col 0: Message type (always SMS — dropped)
- Col 1: Direction (send/receive)
- Col 2: Sender ID
- Col 3: Contact (duplicate of Col 2 — dropped)
- Col 4: Timestamp
- Col 5: Thread/contact ID
- Col 6: Label (SPAM or HAM)
- Col 7+: Message text (spans multiple columns due to commas in SMS content)

In [1]:
import os
import sys
import glob
import pandas as pd

# Add project root to path so src/ imports work
sys.path.append(os.path.abspath('..'))

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DIR       = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

os.makedirs(PROCESSED_DIR, exist_ok=True)

# Verify raw files are present
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.csv')))
print(f'CSV files found: {len(csv_files)}')
for f in csv_files:
    print(f'  {os.path.basename(f)}')

CSV files found: 20
  USER 1.csv
  USER 10.csv
  USER 11.csv
  USER 12.csv
  USER 13.csv
  USER 14.csv
  USER 15.csv
  USER 16.csv
  USER 17.csv
  USER 18.csv
  USER 19.csv
  USER 2.csv
  USER 20.csv
  USER 3.csv
  USER 4.csv
  USER 5.csv
  USER 6.csv
  USER 7.csv
  USER 8.csv
  USER 9.csv


In [3]:
# ── Load and merge all 20 files ───────────────────────────────────────────────
def load_all_users(data_dir):
    """
    Load every USER_*.csv file and return a merged raw DataFrame.
    No headers in source files — all columns read as strings.
    """
    all_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))
    if not all_files:
        raise FileNotFoundError(f'No CSV files found in {data_dir}')

    frames = []
    for f in all_files:
        df = pd.read_csv(
            f,
            header=None,
            dtype=str,
            encoding='utf-8',
            on_bad_lines='skip'
        )
        frames.append(df)
        print(f'  {os.path.basename(f):20s} — {len(df):>4} rows')

    merged = pd.concat(frames, ignore_index=True)
    print(f'\nTotal raw rows merged: {len(merged)}')
    return merged


raw = load_all_users(RAW_DIR)

  USER 1.csv           —   53 rows
  USER 10.csv          —  633 rows
  USER 11.csv          —  243 rows
  USER 12.csv          —  387 rows
  USER 13.csv          —  117 rows
  USER 14.csv          —  227 rows
  USER 15.csv          —  605 rows
  USER 16.csv          — 1082 rows
  USER 17.csv          —  180 rows
  USER 18.csv          —  167 rows
  USER 19.csv          —  626 rows
  USER 2.csv           —  130 rows
  USER 20.csv          —   29 rows
  USER 3.csv           —   23 rows
  USER 4.csv           —  182 rows
  USER 5.csv           —  122 rows
  USER 6.csv           —  116 rows
  USER 7.csv           —  235 rows
  USER 8.csv           —   34 rows
  USER 9.csv           —   49 rows

Total raw rows merged: 5240


In [4]:
# ── Inspect raw structure ─────────────────────────────────────────────────────
print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
print()
raw.head(10)

Shape: (5240, 28)
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]



,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
0,SMS,send,127,127,04/09/2014 08:29,1026,SPAM,CANCLE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SMS,receive,127,127,04/09/2014 08:25,1026,SPAM,Your plan Smallie.is going to be renewed. Plea...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SMS,receive,127,127,04/09/2014 08:25,1026,SPAM,Dear Glo subscriber,you just passed 80% of your Plan Validity Per...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SMS,receive,127,127,01/09/2014 14:25,1026,SPAM,Welcome to Smallie. It will expire on 04/09/20...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SMS,send,127,127,25/08/2014 10:13,1026,SPAM,Info,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SMS,send,127,127,25/08/2014 07:47,1026,SPAM,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SMS,receive,127,127,25/08/2014 07:38,1026,SPAM,Please send :1 for Day Plans2 for Week Plans...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SMS,send,127,127,25/08/2014 07:38,1026,SPAM,ACTIVATE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,SMS,receive,127,127,25/08/2014 07:37,1026,SPAM,Please send ACTIVATE to 127 for a list of options,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SMS,send,127,127,25/08/2014 07:37,1026,SPAM,PAYYOU,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# ── Extract clean records ─────────────────────────────────────────────────────
def build_clean_dataframe(raw):
    """
    Extract label (col 6) and message text (col 7 onwards rejoined).
    Drops malformed rows where label is not SPAM or HAM.
    Drops rows with empty message content after extraction.
    """
    records = []
    skipped = 0

    for _, row in raw.iterrows():
        # Extract and validate label
        label_raw = str(row.iloc[6]).strip().upper() if pd.notna(row.iloc[6]) else ''
        if label_raw not in ('SPAM', 'HAM'):
            skipped += 1
            continue

        # Rejoin message text from col 7 onwards
        # This handles commas inside SMS content that broke CSV parsing
        msg_parts = [
            str(x) for x in row.iloc[7:]
            if pd.notna(x) and str(x).strip() not in ('', 'nan')
        ]
        message = ','.join(msg_parts).strip()

        if not message:
            skipped += 1
            continue

        records.append({
            'label':     1 if label_raw == 'SPAM' else 0,
            'label_str': label_raw,
            'message':   message,
            'sender':    str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else '',
            'direction': str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else '',
            'timestamp': str(row.iloc[4]).strip() if pd.notna(row.iloc[4]) else '',
        })

    df = pd.DataFrame(records)
    print(f'Rows extracted : {len(df)}')
    print(f'Rows skipped   : {skipped}')
    return df


df = build_clean_dataframe(raw)

Rows extracted : 5240
Rows skipped   : 0


In [6]:
# ── Remove duplicates ─────────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset='message').reset_index(drop=True)
after  = len(df)
print(f'Duplicates removed : {before - after}')
print(f'Final dataset size : {after} messages')

Duplicates removed : 1337
Final dataset size : 3903 messages


In [7]:
# ── Class distribution ────────────────────────────────────────────────────────
counts = df['label_str'].value_counts()
print('Class Distribution:')
print(f"  HAM  (Legitimate) : {counts.get('HAM',  0):>5}")
print(f"  SPAM (Fraudulent) : {counts.get('SPAM', 0):>5}")
print(f"  Total             : {len(df):>5}")
print(f"\nHAM  % : {counts.get('HAM',  0)/len(df)*100:.1f}%")
print(f"SPAM % : {counts.get('SPAM', 0)/len(df)*100:.1f}%")

Class Distribution:
  HAM  (Legitimate) :  2430
  SPAM (Fraudulent) :  1473
  Total             :  3903

HAM  % : 62.3%
SPAM % : 37.7%


In [8]:
# ── Preview ───────────────────────────────────────────────────────────────────
print('Sample HAM messages:')
display(df[df['label']==0][['label_str','message']].head(3))

print('\nSample SPAM messages:')
display(df[df['label']==1][['label_str','message']].head(3))

Sample HAM messages:


,label_str,message
16,HAM,Here's a genuine Birthday prayer from the hear...
17,HAM,A million and one THANKS is nt enough to appre...
21,HAM,"8051416112,this is Yemisi 's no. Sorry 4 the d..."



Sample SPAM messages:


,label_str,message
0,SPAM,CANCLE
1,SPAM,Your plan Smallie.is going to be renewed. Plea...
2,SPAM,"Dear Glo subscriber, you just passed 80% of yo..."


In [9]:
# ── Save to processed/ ────────────────────────────────────────────────────────
out_path = os.path.join(PROCESSED_DIR, 'cleaned_dataset.csv')
df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {df.shape}')

Saved: ../data/processed/cleaned_dataset.csv
Shape: (3903, 6)
